In [1]:
# ============================================================
# 02_data_cleaning.ipynb — NETTOYAGE + VARIABLE CIBLE
# Problématique : prédiction du risque de retard de livraison
# ============================================================

import sys
sys.path.append('..')
from src.utils import *
from math import radians, sin, cos, sqrt, atan2

# -------------------------------------------------------
print("\n" + "="*60)
print("   ÉTAPE 1 — CHARGEMENT DES TABLES")
print("="*60 + "\n")
# -------------------------------------------------------

orders    = pd.read_csv('../data/raw/olist_orders_dataset.csv')
items     = pd.read_csv('../data/raw/olist_order_items_dataset.csv')
products  = pd.read_csv('../data/raw/olist_products_dataset.csv')
customers = pd.read_csv('../data/raw/olist_customers_dataset.csv')
sellers   = pd.read_csv('../data/raw/olist_sellers_dataset.csv')
geo       = pd.read_csv('../data/raw/olist_geolocation_dataset.csv')

print("✓ Tables chargées")


# -------------------------------------------------------
print("\n" + "="*60)
print("   ÉTAPE 2 — NETTOYAGE orders + VARIABLE CIBLE")
print("="*60 + "\n")
# -------------------------------------------------------

# Conversion des dates
date_cols = ['order_purchase_timestamp', 'order_approved_at',
             'order_delivered_carrier_date', 'order_delivered_customer_date',
             'order_estimated_delivery_date']
orders = convertir_dates(orders, date_cols)

# Décision 2 : supprimer les 8 "delivered" sans date
avant = len(orders)
orders = orders[~(
    (orders['order_status'] == 'delivered') &
    (orders['order_delivered_customer_date'].isnull())
)]
print(f"✓ Lignes supprimées (delivered sans date) : {avant - len(orders)}")

# Décision 3 : flag incohérence chronologique
orders['date_incoherente'] = (
    orders['order_delivered_customer_date'] < orders['order_purchase_timestamp']
)
print(f"✓ Commandes avec incohérence livraison < achat : {orders['date_incoherente'].sum()}")
orders = orders[~orders['date_incoherente']]

# Décision 1 : ne garder que les commandes livrées pour le modèle
orders_livrees = orders[orders['order_status'] == 'delivered'].copy()
print(f"✓ Commandes livrées utilisables : {len(orders_livrees)}")

# ⭐⭐ VARIABLE CIBLE : retard en jours
# Positif = livré en retard | Négatif = livré en avance | 0 = pile à l'heure
orders_livrees['retard_jours'] = (
    orders_livrees['order_delivered_customer_date'] -
    orders_livrees['order_estimated_delivery_date']
).dt.days

# Variable cible binaire pour la classification
orders_livrees['est_en_retard'] = (orders_livrees['retard_jours'] > 0).astype(int)

print(f"\n--- Distribution de la variable cible ---")
print(f"  Retard moyen   : {orders_livrees['retard_jours'].mean():.1f} jours")
print(f"  Retard médian  : {orders_livrees['retard_jours'].median():.1f} jours")
print(f"  % en retard    : {orders_livrees['est_en_retard'].mean()*100:.1f}%")
print(f"  % à l'heure    : {(1-orders_livrees['est_en_retard'].mean())*100:.1f}%")

# Autres variables temporelles utiles
orders_livrees['delai_livraison_total'] = (
    orders_livrees['order_delivered_customer_date'] -
    orders_livrees['order_purchase_timestamp']
).dt.days
orders_livrees['delai_approbation_h'] = (
    orders_livrees['order_approved_at'] -
    orders_livrees['order_purchase_timestamp']
).dt.total_seconds() / 3600
orders_livrees['mois_achat'] = orders_livrees['order_purchase_timestamp'].dt.month
orders_livrees['jour_semaine_achat'] = orders_livrees['order_purchase_timestamp'].dt.dayofweek


# -------------------------------------------------------
print("\n" + "="*60)
print("   ÉTAPE 3 — DÉDUPLICATION geolocation")
print("="*60 + "\n")
# -------------------------------------------------------

print(f"Avant : {len(geo):,} lignes")
geo_dedup = geo.groupby('geolocation_zip_code_prefix').agg(
    lat=('geolocation_lat', 'median'),
    lng=('geolocation_lng', 'median')
).reset_index()
geo_dedup.columns = ['zip_code_prefix', 'lat', 'lng']
print(f"Après : {len(geo_dedup):,} lignes")


# -------------------------------------------------------
print("\n" + "="*60)
print("   ÉTAPE 4 — NETTOYAGE items + flag incohérence")
print("="*60 + "\n")
# -------------------------------------------------------

items = convertir_dates(items, ['shipping_limit_date'])
items_merged = items.merge(
    orders_livrees[['order_id', 'order_purchase_timestamp']],
    on='order_id', how='inner'
)
items_merged['shipping_incoherent'] = (
    items_merged['shipping_limit_date'] < items_merged['order_purchase_timestamp']
)
print(f"✓ shipping_limit_date incohérentes : {items_merged['shipping_incoherent'].sum()}")

items_merged['delai_avant_expedition_h'] = (
    items_merged['shipping_limit_date'] - items_merged['order_purchase_timestamp']
).dt.total_seconds() / 3600
items_merged.loc[items_merged['shipping_incoherent'], 'delai_avant_expedition_h'] = np.nan


# -------------------------------------------------------
print("\n" + "="*60)
print("   ÉTAPE 5 — NETTOYAGE products + VOLUME")
print("="*60 + "\n")
# -------------------------------------------------------

# Décision : poids = 0 → imputer par médiane de la catégorie
zero_weight = (products['product_weight_g'] == 0).sum()
print(f"Produits poids = 0 : {zero_weight}")

products['product_category_name'] = products['product_category_name'].fillna('unknown')
mediane_cat = products.groupby('product_category_name')['product_weight_g'].transform('median')
products.loc[products['product_weight_g'] == 0, 'product_weight_g'] = mediane_cat

# Imputation des 2 nulls restants par médiane globale
dims = ['product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']
products = imputer_mediane(products, dims)

# ⭐ Feature dérivée : volume du colis
products['volume_cm3'] = (
    products['product_length_cm'] *
    products['product_height_cm'] *
    products['product_width_cm']
)
print(f"✓ Volume calculé — médiane : {products['volume_cm3'].median():.0f} cm³")


# -------------------------------------------------------
print("\n" + "="*60)
print("   ÉTAPE 6 — CALCUL DE LA DISTANCE (Haversine)")
print("   Feature géographique clé du modèle")
print("="*60 + "\n")
# -------------------------------------------------------

def haversine(lat1, lon1, lat2, lon2):
    """Distance en km entre 2 points GPS."""
    R = 6371  # rayon terre en km
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1-a))

# Merger coordonnées customer
customers_geo = customers.merge(
    geo_dedup, left_on='customer_zip_code_prefix',
    right_on='zip_code_prefix', how='left'
).rename(columns={'lat': 'cust_lat', 'lng': 'cust_lng'})

# Merger coordonnées seller
sellers_geo = sellers.merge(
    geo_dedup, left_on='seller_zip_code_prefix',
    right_on='zip_code_prefix', how='left'
).rename(columns={'lat': 'seller_lat', 'lng': 'seller_lng'})

print(f"✓ Customers sans coordonnées : {customers_geo['cust_lat'].isnull().sum()} "
      f"({customers_geo['cust_lat'].isnull().mean()*100:.1f}%)")
print(f"✓ Sellers sans coordonnées   : {sellers_geo['seller_lat'].isnull().sum()} "
      f"({sellers_geo['seller_lat'].isnull().mean()*100:.1f}%)")

# Table finale : order_id + customer_geo + seller_geo
base = items_merged[['order_id', 'seller_id', 'product_id',
                      'delai_avant_expedition_h']].merge(
    orders_livrees[['order_id', 'customer_id', 'retard_jours',
                     'est_en_retard', 'delai_livraison_total',
                     'mois_achat', 'jour_semaine_achat']],
    on='order_id', how='inner'
)
base = base.merge(customers_geo[['customer_id', 'customer_state',
                                  'cust_lat', 'cust_lng']],
                   on='customer_id', how='left')
base = base.merge(sellers_geo[['seller_id', 'seller_state',
                                'seller_lat', 'seller_lng']],
                   on='seller_id', how='left')
base = base.merge(products[['product_id', 'volume_cm3',
                             'product_weight_g']],
                   on='product_id', how='left')

# Calcul distance pour les lignes complètes
mask_complet = base[['cust_lat','cust_lng','seller_lat','seller_lng']].notnull().all(axis=1)
base['distance_km'] = np.nan
base.loc[mask_complet, 'distance_km'] = base[mask_complet].apply(
    lambda r: haversine(r['cust_lat'], r['cust_lng'],
                        r['seller_lat'], r['seller_lng']), axis=1
)

base['meme_etat'] = (base['customer_state'] == base['seller_state']).astype(int)

print(f"\n✓ Distance calculée pour {mask_complet.sum()} lignes ({mask_complet.mean()*100:.1f}%)")
print(f"  Distance moyenne  : {base['distance_km'].mean():.0f} km")
print(f"  Distance médiane  : {base['distance_km'].median():.0f} km")
print(f"  % même état       : {base['meme_etat'].mean()*100:.1f}%")

# -------------------------------------------------------
print("\n" + "="*60)
print("   ÉTAPE 7 — TRAITEMENT DES ANOMALIES DÉTECTÉES")
print("   (avant export — sur la table base déjà fusionnée)")
print("="*60 + "\n")
# -------------------------------------------------------

# 1. Erreurs géographiques impossibles (Brésil max ~4400 km)
avant = len(base)
base['erreur_geo'] = base['distance_km'] > 4400
print(f"✓ Distances impossibles (>4400km) : {base['erreur_geo'].sum()}")

# 2. Erreurs de délai d'expédition aberrant
base['erreur_delai'] = base['delai_avant_expedition_h'] > 2000
print(f"✓ Délais expédition aberrants (>2000h) : {base['erreur_delai'].sum()}")

# Exclusion des erreurs certaines
base = base[~base['erreur_geo'] & ~base['erreur_delai']].copy()
base = base.drop(columns=['erreur_geo', 'erreur_delai'])
print(f"✓ Lignes supprimées : {avant - len(base)}")

# 3. Isolation Forest — feature prédictive
from sklearn.ensemble import IsolationForest
colonnes_continues = ['distance_km', 'volume_cm3', 'product_weight_g',
                       'delai_avant_expedition_h', 'delai_livraison_total']

subset = base[colonnes_continues].dropna()
iso = IsolationForest(contamination=0.02, random_state=42, n_jobs=-1)
base.loc[subset.index, 'anomalie_multivariee'] = (
    iso.fit_predict(subset) == -1
).astype(int)
base['anomalie_multivariee'] = base['anomalie_multivariee'].fillna(0).astype(int)

n_anom = base['anomalie_multivariee'].sum()
print(f"✓ Feature 'anomalie_multivariee' créée : {n_anom} lignes ({n_anom/len(base)*100:.1f}%)")


# -------------------------------------------------------
print("\n" + "="*60)
print("   ÉTAPE 8 — EXPORT FINAL")
print("="*60 + "\n")
# -------------------------------------------------------

import os
os.makedirs('../data/processed', exist_ok=True)
base.to_csv('../data/processed/logistique_base.csv', index=False)
orders_livrees.to_csv('../data/processed/orders_clean.csv', index=False)
products.to_csv('../data/processed/products_clean.csv', index=False)
geo_dedup.to_csv('../data/processed/geo_dedup.csv', index=False)

print(f"✓ logistique_base.csv exporté : {base.shape}")
print(f"✓ Colonnes : {base.columns.tolist()}")
print(f"\n→ Prochain : 03_eda.ipynb")


   ÉTAPE 1 — CHARGEMENT DES TABLES

✓ Tables chargées

   ÉTAPE 2 — NETTOYAGE orders + VARIABLE CIBLE

✓ 5 colonne(s) converties en datetime
✓ Lignes supprimées (delivered sans date) : 8
✓ Commandes avec incohérence livraison < achat : 0
✓ Commandes livrées utilisables : 96470

--- Distribution de la variable cible ---
  Retard moyen   : -11.9 jours
  Retard médian  : -12.0 jours
  % en retard    : 6.8%
  % à l'heure    : 93.2%

   ÉTAPE 3 — DÉDUPLICATION geolocation

Avant : 1,000,163 lignes
Après : 19,015 lignes

   ÉTAPE 4 — NETTOYAGE items + flag incohérence

✓ 1 colonne(s) converties en datetime
✓ shipping_limit_date incohérentes : 0

   ÉTAPE 5 — NETTOYAGE products + VOLUME

Produits poids = 0 : 4
✓ product_weight_g                    imputé avec médiane = 700.00
✓ product_length_cm                   imputé avec médiane = 25.00
✓ product_height_cm                   imputé avec médiane = 13.00
✓ product_width_cm                    imputé avec médiane = 20.00
✓ Volume calculé — mé